# Medical Domain | QLoRA Fine-Tuning
**Kaggle T4 GPU** | Merged dataset | Pre/Post per-dataset evaluation

Change `MODEL_ID` in Cell 3 to switch between the three models:
- `"google/gemma-3-270m-it"`
- `"HuggingFaceTB/SmolLM2-360M-Instruct"`
- `"Qwen/Qwen3-0.6B"`

## Cell 1 — Install

In [ ]:
%%capture
!pip install -q "transformers>=4.50.0"
!pip install -q "tokenizers>=0.21.0"
!pip install -q "huggingface_hub>=0.23.0"
!pip install -q "peft>=0.11.0"
!pip install -q "trl>=0.9.0"
!pip install -q "bitsandbytes>=0.43.0"
!pip install -q "accelerate>=0.30.0"
!pip install -q "datasets>=2.19.0"
!pip install -q sacrebleu==2.4.3
!pip install -q rouge-score==0.1.2
!pip install -q scikit-learn
!pip install -q nltk
!pip install -q sentencepiece
print("Done")

## Cell 2 — Imports & GPU Check

In [ ]:
import socket, os, gc, csv, json, time, random, logging, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from typing import Dict, List
from collections import Counter

warnings.filterwarnings("ignore")
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainerCallback, TrainerState, TrainerControl,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel, TaskType
from trl import SFTTrainer, SFTConfig

import sacrebleu
from rouge_score import rouge_scorer as rouge_scorer_lib
from sklearn.metrics import f1_score
import nltk
from nltk.tokenize import word_tokenize
nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Logger
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger("medical_ft")

log.info(f"torch        : {torch.__version__}")
log.info(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    log.info(f"GPU          : {torch.cuda.get_device_name(0)}")
    log.info(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

import transformers, peft, trl
log.info(f"transformers : {transformers.__version__}")
log.info(f"peft         : {peft.__version__}")
log.info(f"trl          : {trl.__version__}")
!nvidia-smi
print("Imports OK")

## Cell 3 — Config
> **Change `MODEL_ID` here to switch models.**  
> Accept the model licence on HuggingFace and add your token to **Kaggle → Add-ons → Secrets → HF_TOKEN**.

In [ ]:
# Paths
WORK        = Path("/kaggle/working")
LOG_DIR     = WORK / "logs";          LOG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = WORK / "results";       RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR    = WORK / "checkpoints";   CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Model — change this line to run a different model:
#   "google/gemma-3-270m-it"
#   "HuggingFaceTB/SmolLM2-360M-Instruct"
#   "Qwen/Qwen3-0.6B"
MODEL_ID    = "google/gemma-3-270m-it"
MAX_SEQ_LEN = 512

# Dataset sample sizes (report §4)
N_PUBMEDQA  = 1000
N_USMLE     = 4000
N_ALPACA    = 8000
N_COMPMEDQA = 3000

# Comprehensive Medical Q&A
# Option A: attach the Kaggle dataset (thedevastator/comprehensive-medical-qa)
# Option B (automatic fallback): loads from HuggingFace if CSV not found
COMPMEDQA_CSV = "/kaggle/input/comprehensive-medical-qa/train.csv"

# QLoRA — matches report Table 2
BNB_CFG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# LoRA adapters attached to query and value projections only (report Table 2)
LORA_CFG = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Training hyperparameters
TRAIN_CFG = dict(
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=1,
    max_grad_norm=0.3,
    weight_decay=0.001,
    report_to="none",
    dataloader_num_workers=2,
)

# Inference hyperparameters — matches report Appendix A (Table 12)
N_EVAL_SAMPLES  = 200
MAX_NEW_TOKENS  = 150
TEMPERATURE     = 0.3
TOP_P           = 0.4
REP_PENALTY     = 1.2

ROUGE = rouge_scorer_lib.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

log.info("Config ready.")

## Cell 4 — HuggingFace Login

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
login(token=secrets.get_secret("HF_TOKEN"), add_to_git_credential=False)
log.info("HuggingFace login OK")

## Cell 5 — Dataset Loaders & Prompt Format

All samples use the unified 4-tag schema.

In [ ]:
def fmt(instruction: str, context: str, inp: str, out: str) -> str:
    """Unified 4-tag prompt format (report §3.3)."""
    return (
        f"<instruction>{instruction.strip()}</instruction>\n"
        f"<context>{context.strip()}</context>\n"
        f"<input>{inp.strip()}</input>\n"
        f"<output>{out.strip()}</output>"
    )


def load_pubmedqa() -> List[Dict]:
    """Report §4.1.1: PubMedQA - Biomedical QA with yes/no/maybe decision."""
    log.info("Loading PubMedQA ...")
    ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled",
                      split="train", trust_remote_code=True)
    ds = ds.shuffle(seed=SEED).select(range(min(N_PUBMEDQA, len(ds))))
    out = []
    for r in ds:
        ctx = " ".join((r["context"].get("contexts") or [])[:3]) or "N/A"
        ans = f"{r['long_answer']} Decision: {r['final_decision']}.".strip()
        out.append({
            "text": fmt(
                "Based on the provided biomedical context, answer the question and conclude with yes, no, or maybe.",
                ctx, r["question"], ans),
            "answer": ans, "source": "pubmedqa"
        })
    log.info(f"  PubMedQA      : {len(out)} samples")
    return out


def load_usmle() -> List[Dict]:
    """Report §4.1.2: MedQA-USMLE - Clinical multiple-choice question answering (4-option)."""
    log.info("Loading MedQA-USMLE ...")
    ds = load_dataset("GBaker/MedQA-USMLE-4-options",
                      split="train", trust_remote_code=True)
    ds = ds.shuffle(seed=SEED).select(range(min(N_USMLE, len(ds))))
    out = []
    for r in ds:
        opts = r["options"]
        ctx  = "\n".join(f"{k}. {v}" for k, v in opts.items())
        key  = r.get("answer_idx", "")
        ans  = f"{key}. {opts.get(key, r.get('answer', ''))}"
        out.append({
            "text": fmt(
                "Answer the following USMLE-style clinical multiple-choice question. State the correct option letter and briefly justify.",
                ctx, r["question"], ans),
            "answer": ans, "source": "medqa_usmle"
        })
    log.info(f"  MedQA-USMLE   : {len(out)} samples")
    return out


def load_alpaca() -> List[Dict]:
    """Report §4.2.1: AlpaCare-MedInstruct - Medical instruction following."""
    log.info("Loading AlpaCare-MedInstruct-52k ...")
    ds = load_dataset("lavita/AlpaCare-MedInstruct-52k",
                      split="train", trust_remote_code=True)
    ds = ds.shuffle(seed=SEED).select(range(min(N_ALPACA, len(ds))))
    out = []
    for r in ds:
        ans = (r.get("output") or "").strip()
        if not ans:
            continue
        out.append({
            "text": fmt(
                r.get("instruction") or "Provide a helpful medical response.",
                "Open-ended medical instruction following.",
                r.get("input") or "N/A", ans),
            "answer": ans, "source": "alpaca_care"
        })
    log.info(f"  AlpaCare      : {len(out)} samples")
    return out


def load_compmedqa() -> List[Dict]:
    """Report §4.3.1: Comprehensive Medical Q&A - General medical question answering.
    
    Primary source: Kaggle dataset (thedevastator/comprehensive-medical-q-a-dataset)
    Fallback: HuggingFace medical_meadow_medical_flashcards if Kaggle dataset not attached
    """
    p = Path(COMPMEDQA_CSV)
    if p.exists():
        log.info("Loading Comprehensive Medical Q&A from Kaggle CSV ...")
        df = pd.read_csv(p)
        df.columns = [c.strip().lower() for c in df.columns]
        qc = next((c for c in df.columns if "question" in c), None)
        ac = next((c for c in df.columns if "answer"   in c), None)
        if qc and ac:
            df = df[[qc, ac]].dropna().sample(frac=1, random_state=SEED).head(N_COMPMEDQA)
            out = []
            for _, row in df.iterrows():
                q, a = str(row[qc]).strip(), str(row[ac]).strip()
                out.append({
                    "text": fmt("Answer the following medical question clearly and accurately.",
                                "General medical knowledge — anatomy, pharmacology, pathology, clinical medicine.", q, a),
                    "answer": a, "source": "comprehensive_medqa"
                })
            log.info(f"  Comprehensive : {len(out)} samples (Kaggle CSV)")
            return out
        log.warning("Cannot find question/answer columns — falling back to HuggingFace.")
    
    # HuggingFace fallback
    log.info("Loading Comprehensive Medical Q&A from HuggingFace (fallback) ...")
    ds = load_dataset("rungalileo/medical_meadow_medical_flashcards",
                      split="train", trust_remote_code=True)
    ds = ds.shuffle(seed=SEED).select(range(min(N_COMPMEDQA, len(ds))))
    out = []
    for r in ds:
        q = str(r.get("input", "")).strip()
        a = str(r.get("output", "")).strip()
        if not q or not a:
            continue
        out.append({
            "text": fmt("Answer the following medical question clearly and accurately.",
                        "General medical knowledge — anatomy, pharmacology, pathology, clinical medicine.", q, a),
            "answer": a, "source": "comprehensive_medqa"
        })
    log.info(f"  Comprehensive : {len(out)} samples (HuggingFace fallback)")
    return out


def load_mednli() -> List[Dict]:
    """Report §4.4.1: MedNLI - Clinical Natural Language Inference.
    
    Held-out evaluation only, NOT included in training corpus.
    Uses test split for evaluation as this dataset is not in the training set.
    """
    log.info("Loading MedNLI (held-out evaluation) ...")
    ds = load_dataset("bigbio/mednli", "mednli_bigbio_text",
                      split="test", trust_remote_code=True)
    out = []
    for r in ds:
        premise    = str(r.get("text_1", "")).strip()
        hypothesis = str(r.get("text_2", "")).strip()
        label      = str(r.get("label",  "")).strip().lower()
        if not premise or not hypothesis or not label:
            continue
        out.append({
            "text": fmt(
                "Classify the relationship between the premise and hypothesis as "
                "entailment, neutral, or contradiction. Respond with exactly one word.",
                f"Premise: {premise}", f"Hypothesis: {hypothesis}", label),
            "answer": label, "source": "mednli", "is_classification": True
        })
    log.info(f"  MedNLI        : {len(out)} samples (held-out, not in training)")
    return out


def load_meqsum() -> List[Dict]:
    """Report §4.5.1: MeQSum - Consumer Health Question Summarisation.
    
    Held-out evaluation only, NOT included in training corpus.
    """
    log.info("Loading MeQSum (held-out evaluation) ...")
    ds = load_dataset("GAIA-benchmark/MeQSum", split="train", trust_remote_code=True)
    out = []
    for r in ds:
        chq     = str(r.get("CHQ",     "")).strip()
        summary = str(r.get("Summary", "")).strip()
        if not chq or not summary:
            continue
        out.append({
            "text": fmt(
                "Summarise the following consumer health question into a concise, "
                "clinically focused reformulation.",
                "Consumer health question summarisation.", chq[:1000], summary),
            "answer": summary, "source": "meqsum"
        })
    log.info(f"  MeQSum        : {len(out)} samples (held-out, not in training)")
    return out


def load_bioasq() -> List[Dict]:
    """Report §4.1.3: BioASQ - Biomedical Factoid QA (UNSEEN BENCHMARK).
    
    This is the UNSEEN evaluation benchmark - NOT included in training.
    Tests generalization to out-of-distribution clinical QA task (report §1.1, §6.2.3).
    """
    log.info("Loading BioASQ (UNSEEN BENCHMARK - not in training) ...")
    ds = load_dataset("bigbio/bioasq", "bioasq_10b_bigbio_qa",
                      split="test", trust_remote_code=True)
    out = []
    for r in ds:
        q        = str(r.get("question", "")).strip()
        ans_field = r.get("answer", {})
        if isinstance(ans_field, dict):
            ans = " ".join(ans_field.get("text", [])) or "N/A"
        elif isinstance(ans_field, list):
            ans = " ".join(str(a) for a in ans_field) or "N/A"
        else:
            ans = str(ans_field).strip() or "N/A"
        ctx = str(r.get("context", "Biomedical factoid QA grounded in PubMed research.")).strip()
        if not q:
            continue
        out.append({
            "text": fmt(
                "Answer the following biomedical question based on PubMed literature. "
                "Be precise and concise.",
                ctx[:800], q, ans),
            "answer": ans, "source": "bioasq"
        })
    log.info(f"  BioASQ        : {len(out)} samples (UNSEEN BENCHMARK)")
    return out


print("Dataset loader functions defined.")

## Cell 6 — Build Combined Training Dataset (80/20 split)


The 80/20 split is applied per source so each dataset is represented in the test set (report Table 2).

In [ ]:
# Training corpus (4 datasets, report §3.4)
train_source_samples: List[Dict] = []
train_source_samples.extend(load_pubmedqa())
train_source_samples.extend(load_usmle())
train_source_samples.extend(load_alpaca())
train_source_samples.extend(load_compmedqa())

# Held-out evaluation datasets (not included in training)
eval_only_samples: Dict[str, List[Dict]] = {
    "mednli": load_mednli(),
    "meqsum": load_meqsum(),
    "bioasq": load_bioasq(),
}

# 80/20 split per source so each dataset has test samples (report Table 2)
per_source_train: List[Dict] = []
per_source_test:  List[Dict] = []
src_groups: Dict[str, List[Dict]] = {}
for s in train_source_samples:
    src_groups.setdefault(s["source"], []).append(s)
for src, samps in src_groups.items():
    random.shuffle(samps)
    n_tr = int(len(samps) * 0.8)
    per_source_train.extend(samps[:n_tr])
    per_source_test.extend(samps[n_tr:])

train_samples = per_source_train
test_samples  = per_source_test

# Summary
print("\n" + "="*60)
print(f"  Training corpus total : {len(train_source_samples)}")
print(f"  Train split (80%)     : {len(train_samples)}")
print(f"  Test split (20%)      : {len(test_samples)}")
print("  --- per source (report Table 2: 80% train / 20% test) ---")
for src, samps in src_groups.items():
    n_tr = int(len(samps) * 0.8)
    print(f"  {src:<30}: {n_tr:>5} train / {len(samps)-n_tr:>5} test")
print()
print("  --- held-out evaluation datasets (NOT in training) ---")
for name, samps in eval_only_samples.items():
    print(f"  {name:<30}: {len(samps):>5} (eval-only)")
print("="*60)

# HuggingFace Dataset for SFTTrainer (text column only)
train_hf = Dataset.from_dict({"text": [s["text"] for s in train_samples]})

# Persist to /kaggle/working
import shutil
KAGGLE_OUTPUT = WORK / "medical_ft_data"
KAGGLE_OUTPUT.mkdir(parents=True, exist_ok=True)
train_hf.save_to_disk(str(KAGGLE_OUTPUT / "train_hf"))
with open(KAGGLE_OUTPUT / "test_samples.json", "w") as f:
    json.dump(test_samples, f, indent=2)
with open(KAGGLE_OUTPUT / "eval_only_samples.json", "w") as f:
    json.dump({k: v for k, v in eval_only_samples.items()}, f, indent=2)
log.info(f"Datasets saved to {KAGGLE_OUTPUT}")

## Cell 7 — Metric Functions

Macro-F1 is computed for MedNLI only

In [ ]:
def token_f1(pred: str, ref: str) -> float:
    """SQuAD-style token-overlap F1 (report §5.1)."""
    p_toks = set(word_tokenize(pred.lower()))
    r_toks = set(word_tokenize(ref.lower()))
    if not p_toks or not r_toks:
        return 0.0
    common = p_toks & r_toks
    if not common:
        return 0.0
    prec = len(common) / len(p_toks)
    rec  = len(common) / len(r_toks)
    return 2 * prec * rec / (prec + rec)


def compute_metrics(predictions: List[str], references: List[str],
                    is_classification: bool = False) -> Dict:
    """Compute BLEU (0-100), ROUGE-1/2/L, Token-F1, and optionally Macro-F1.
    
    Report §5.1: Evaluation Metrics
    - BLEU: 0-100 scale (SacreBLEU)
    - ROUGE-1/2/L: [0, 1] scale
    - Token F1: [0, 1] scale
    - Macro-F1: [0, 1] scale (MedNLI only)
    """
    pairs = [(p.strip(), r.strip())
             for p, r in zip(predictions, references)
             if p.strip() and r.strip()]
    if not pairs:
        zero = {"bleu": 0.0, "rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0, "f1": 0.0, "n": 0}
        if is_classification:
            zero["macro_f1"] = 0.0
        return zero

    preds, refs = zip(*pairs)

    # BLEU — 0-100 scale, matches report Tables 4-10
    try:
        bleu = sacrebleu.corpus_bleu(list(preds), [list(refs)]).score
    except Exception:
        bleu = 0.0

    # ROUGE
    r1s, r2s, rLs = [], [], []
    for p, r in zip(preds, refs):
        sc = ROUGE.score(r, p)
        r1s.append(sc["rouge1"].fmeasure)
        r2s.append(sc["rouge2"].fmeasure)
        rLs.append(sc["rougeL"].fmeasure)

    # Token F1
    f1s = [token_f1(p, r) for p, r in zip(preds, refs)]

    results = {
        "bleu":   round(bleu,          4),
        "rouge1": round(np.mean(r1s),  4),
        "rouge2": round(np.mean(r2s),  4),
        "rougeL": round(np.mean(rLs),  4),
        "f1":     round(np.mean(f1s),  4),
        "n":      len(pairs),
    }

    if is_classification:
        # MedNLI: unweighted mean per-class F1 (report §5.1, Table 9)
        try:
            macro_f1 = f1_score(
                list(refs), list(preds), average="macro",
                labels=["entailment", "neutral", "contradiction"],
                zero_division=0)
        except Exception:
            macro_f1 = 0.0
        results["macro_f1"] = round(macro_f1, 4)

    return results


def extract_prompt(text: str) -> str:
    """Return the prompt up to and including the open <output> tag."""
    return text[:text.index("<output>") + len("<output>")] if "<output>" in text else text


def extract_generated(text: str) -> str:
    """Extract text after <output> (stop at </output> if present)."""
    if "<output>" in text:
        after = text[text.index("<output>") + len("<output>"):]
        return after[:after.index("</output>")].strip() if "</output>" in after else after.strip()
    return text.strip()


print("Metric functions defined.")

## Cell 8 — Model Loader & Training Logger

In [ ]:
# CSV logger
TRAIN_LOG_PATH = LOG_DIR / "training_log.csv"
EVAL_LOG_PATH  = LOG_DIR / "eval_log.csv"

def _init_csv(path: Path, fields: List[str]):
    if not path.exists():
        with open(path, "w", newline="") as f:
            csv.DictWriter(f, fieldnames=fields).writeheader()

def _append_csv(path: Path, fields: List[str], row: Dict):
    row["ts"] = datetime.now().strftime("%H:%M:%S")
    with open(path, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=fields).writerow(
            {k: row.get(k, "") for k in fields})

TRAIN_FIELDS = ["ts", "step", "epoch", "loss", "lr", "elapsed_s"]
EVAL_FIELDS  = ["ts", "phase", "dataset", "bleu", "rouge1", "rouge2",
                "rougeL", "f1", "macro_f1", "n"]
_init_csv(TRAIN_LOG_PATH, TRAIN_FIELDS)
_init_csv(EVAL_LOG_PATH,  EVAL_FIELDS)


class StepLogger(TrainerCallback):
    def __init__(self):
        self._t0 = time.time()
    def on_log(self, args, state: TrainerState, control: TrainerControl, logs=None, **kw):
        if logs and "loss" in logs:
            _append_csv(TRAIN_LOG_PATH, TRAIN_FIELDS, {
                "step":      state.global_step,
                "epoch":     round(state.epoch or 0, 3),
                "loss":      round(logs["loss"], 5),
                "lr":        logs.get("learning_rate", ""),
                "elapsed_s": round(time.time() - self._t0, 1),
            })
            log.info(f"step {state.global_step:>5} | "
                     f"loss {logs['loss']:.4f} | "
                     f"epoch {state.epoch:.2f}")


def load_model_and_tokenizer(lora: bool = False):
    """Load MODEL_ID in 4-bit QLoRA mode; optionally attach LoRA adapters.
    
    Report §3.1: Hardware and Environment
    Report §3.2: QLoRA Configuration (Table 2)
    """
    log.info(f"Loading {MODEL_ID} (lora={lora}) ...")
    tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    model_kwargs = {
        "quantization_config": BNB_CFG,
        "device_map": "auto",
        "trust_remote_code": True,
        "torch_dtype": torch.bfloat16,
    }
    # Disable chain-of-thought for Qwen3 (report §2.2)
    if "qwen" in MODEL_ID.lower():
        model_kwargs["enable_thinking"] = False

    mdl = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
    mdl.config.use_cache = False

    if lora:
        mdl = prepare_model_for_kbit_training(mdl, use_gradient_checkpointing=True)
        mdl = get_peft_model(mdl, LORA_CFG)
        mdl.print_trainable_parameters()

    log.info(f"GPU after load: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    return mdl, tok


def free(model=None):
    if model is not None:
        del model
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    log.info(f"GPU after free: {torch.cuda.memory_allocated()/1e9:.2f} GB")


print("Model loader and logger defined.")

## Cell 9 — Per-Dataset Evaluation Runner

Each dataset is evaluated independently, producing the per-dataset rows.

In [ ]:
@torch.no_grad()
def run_evaluation_dataset(model, tokenizer, samples: List[Dict],
                            dataset_name: str, phase: str,
                            n: int = N_EVAL_SAMPLES) -> Dict:
    """Evaluate on up to `n` samples from a single dataset.
    
    Report Appendix A (Table 12): Inference Hyperparameters
    - Max new tokens: 150
    - Temperature: 0.3
    - Top-p: 0.4
    - Repetition penalty: 1.2
    
    Returns a metric dict keyed by metric name.
    """
    model.eval()
    is_clf = any(s.get("is_classification", False) for s in samples)
    subset = random.sample(samples, min(n, len(samples)))
    preds, refs = [], []

    log.info(f"  [{phase}] {dataset_name}: {len(subset)} samples ...")

    for i, s in enumerate(subset):
        prompt = extract_prompt(s["text"])
        ref    = s["answer"]

        enc = tokenizer(
            prompt, return_tensors="pt", truncation=True,
            max_length=MAX_SEQ_LEN - MAX_NEW_TOKENS,
        ).to(model.device)

        # Logit sanity check
        try:
            logits = model(**enc).logits
            if torch.isnan(logits).any() or torch.isinf(logits).any():
                log.warning(f"    Sample {i}: NaN/Inf logits — skipping.")
                continue
        except Exception as e:
            log.warning(f"    Sample {i}: forward pass error ({e}) — skipping.")
            continue

        # Inference hyperparameters — report Appendix A (Table 12)
        gen = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            repetition_penalty=REP_PENALTY,
            pad_token_id=tokenizer.eos_token_id,
        )
        new_ids  = gen[0][enc["input_ids"].shape[1]:]
        raw_text = tokenizer.decode(new_ids, skip_special_tokens=True)
        pred     = extract_generated(raw_text)

        # MedNLI: normalise output to the closest valid label string
        if is_clf:
            pred_lower = pred.lower().strip()
            matched = "neutral"  # default fallback
            for label in ["entailment", "contradiction", "neutral"]:
                if label in pred_lower:
                    matched = label
                    break
            pred = matched

        preds.append(pred)
        refs.append(ref)

        if (i + 1) % 50 == 0:
            log.info(f"    {i+1}/{len(subset)} done")

    metrics = compute_metrics(preds, refs, is_classification=is_clf)
    metrics["phase"]   = phase
    metrics["dataset"] = dataset_name

    _append_csv(EVAL_LOG_PATH, EVAL_FIELDS, metrics)

    mf1_str = f"  MacroF1={metrics['macro_f1']:.4f}" if "macro_f1" in metrics else ""
    log.info(
        f"  [{phase}] {dataset_name}: "
        f"BLEU={metrics['bleu']:.4f}  "
        f"R1={metrics['rouge1']:.4f}  R2={metrics['rouge2']:.4f}  "
        f"RL={metrics['rougeL']:.4f}  F1={metrics['f1']:.4f}"
        + mf1_str
    )
    return metrics


def run_all_datasets(model, tokenizer, phase: str) -> Dict[str, Dict]:
    """Evaluate across all 7 datasets.
    
    Training-source datasets use the held-out 20% test split;
    eval-only datasets use their full loaded sets.
    
    Report §6.1: Overview
    - Before fine-tuning (Benchmark A - zero-shot)
    - After fine-tuning (Benchmark B - merged QLoRA)
    """
    results = {}

    # Group test_samples by source
    per_source: Dict[str, List[Dict]] = {}
    for s in test_samples:
        per_source.setdefault(s["source"], []).append(s)

    # Training-source datasets (20% test split)
    source_display = {
        "pubmedqa":            "PubMedQA",
        "medqa_usmle":         "MedQA-USMLE",
        "alpaca_care":         "AlpaCare-MedInstruct",
        "comprehensive_medqa": "Comprehensive Medical Q&A",
    }
    for src_key, display_name in source_display.items():
        src_samps = per_source.get(src_key, [])
        if not src_samps:
            log.warning(f"  No test samples for source: {src_key}")
            continue
        results[display_name] = run_evaluation_dataset(
            model, tokenizer, src_samps, display_name, phase)

    # Eval-only datasets (held-out, not in training)
    eval_display = {
        "mednli": "MedNLI",
        "meqsum": "MeQSum",
        "bioasq": "BioASQ",
    }
    for src_key, display_name in eval_display.items():
        samps = eval_only_samples.get(src_key, [])
        if not samps:
            log.warning(f"  No samples for eval-only dataset: {src_key}")
            continue
        results[display_name] = run_evaluation_dataset(
            model, tokenizer, samps, display_name, phase)

    return results


print("Evaluation runner defined.")

## Cell 10 — Pre-Fine-Tuning Evaluation (Benchmark A — Zero-Shot Baseline)

In [ ]:
log.info("="*60)
log.info("BENCHMARK A — PRE-FINETUNING EVALUATION (ZERO-SHOT)")
log.info(f"Model: {MODEL_ID}")
log.info("="*60)

base_model, base_tok = load_model_and_tokenizer(lora=False)

all_pre_metrics = run_all_datasets(base_model, base_tok, phase="pre")

# Summary table
W = 80
print("\n" + "="*W)
print(f" PRE-FINETUNING BASELINE (BENCHMARK A) — {MODEL_ID}")
print("="*W)
print(f"  {'Dataset':<32} {'BLEU':>8} {'R-1':>8} {'R-2':>8} {'R-L':>8} {'TokF1':>8} {'MacF1':>8}")
print("-"*W)
for ds_name, m in all_pre_metrics.items():
    mf1 = f"{m['macro_f1']:.4f}" if "macro_f1" in m else "  —  "
    print(f"  {ds_name:<32} {m['bleu']:>8.4f} {m['rouge1']:>8.4f} "
          f"{m['rouge2']:>8.4f} {m['rougeL']:>8.4f} {m['f1']:>8.4f} {mf1:>8}")
print("="*W)

with open(RESULTS_DIR / "pre_metrics.json", "w") as f:
    json.dump(all_pre_metrics, f, indent=2)

free(base_model)
del base_model, base_tok
gc.collect()
print("Base model unloaded.")

## Cell 11 — QLoRA Fine-Tuning (Benchmark B)

In [ ]:
log.info("="*60)
log.info("BENCHMARK B — QLORA FINE-TUNING (MERGED DATASET)")
log.info(f"Training samples : {len(train_hf)}")
log.info("="*60)

ADAPTER_DIR = str(CKPT_DIR / "lora_adapter")

gc.collect()
torch.cuda.empty_cache()

ft_model, ft_tok = load_model_and_tokenizer(lora=True)

sft_config = SFTConfig(
    output_dir=str(CKPT_DIR / "sft_output"),
    num_train_epochs=TRAIN_CFG["num_train_epochs"],
    per_device_train_batch_size=TRAIN_CFG["per_device_train_batch_size"],
    gradient_accumulation_steps=TRAIN_CFG["gradient_accumulation_steps"],
    gradient_checkpointing=True,
    learning_rate=TRAIN_CFG["learning_rate"],
    warmup_ratio=TRAIN_CFG["warmup_ratio"],
    lr_scheduler_type=TRAIN_CFG["lr_scheduler_type"],
    logging_steps=TRAIN_CFG["logging_steps"],
    save_strategy=TRAIN_CFG["save_strategy"],
    save_total_limit=TRAIN_CFG["save_total_limit"],
    fp16=True,  # Report §8: T4 GPU does not support bfloat16 natively
    optim="paged_adamw_8bit",
    max_grad_norm=TRAIN_CFG["max_grad_norm"],
    weight_decay=TRAIN_CFG["weight_decay"],
    report_to=TRAIN_CFG["report_to"],
    dataloader_num_workers=TRAIN_CFG["dataloader_num_workers"],
    max_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,
    seed=SEED,
)

trainer = SFTTrainer(
    model=ft_model,
    processing_class=ft_tok,
    train_dataset=train_hf,
    args=sft_config,
    callbacks=[StepLogger()],
)

t0 = time.time()
result = trainer.train()
elapsed = time.time() - t0

log.info(f"Training finished in {elapsed/60:.1f} min")
log.info(f"Final loss : {result.training_loss:.4f}")

# Save LoRA adapter
ft_model.save_pretrained(ADAPTER_DIR)
ft_tok.save_pretrained(ADAPTER_DIR)
log.info(f"LoRA adapter saved → {ADAPTER_DIR}")

## Cell 12 — Post-Fine-Tuning Evaluation (Benchmark B — Merged Fine-Tuning)

Models are jointly fine-tuned on the combined training datasets, and then evaluated.

In [ ]:
log.info("="*60)
log.info("BENCHMARK B — POST-FINETUNING EVALUATION (MERGED)")
log.info(f"Model: {MODEL_ID}")
log.info("="*60)

all_post_metrics = run_all_datasets(ft_model, ft_tok, phase="post")

# Summary table
W = 80
print("\n" + "="*W)
print(f" POST-FINETUNING (BENCHMARK B) — {MODEL_ID}")
print("="*W)
print(f"  {'Dataset':<32} {'BLEU':>8} {'R-1':>8} {'R-2':>8} {'R-L':>8} {'TokF1':>8} {'MacF1':>8}")
print("-"*W)
for ds_name, m in all_post_metrics.items():
    mf1 = f"{m['macro_f1']:.4f}" if "macro_f1" in m else "  —  "
    print(f"  {ds_name:<32} {m['bleu']:>8.4f} {m['rouge1']:>8.4f} "
          f"{m['rouge2']:>8.4f} {m['rougeL']:>8.4f} {m['f1']:>8.4f} {mf1:>8}")
print("="*W)

with open(RESULTS_DIR / "post_metrics.json", "w") as f:
    json.dump(all_post_metrics, f, indent=2)

free(ft_model)
del ft_model, trainer, ft_tok
gc.collect()
print("Fine-tuned model unloaded.")

## Cell 13 — Results Comparison (Pre vs Post Fine-Tuning)

Compare Benchmark A (zero-shot) vs Benchmark B (merged fine-tuning) to assess adaptation impact.

In [ ]:
METRICS_DISPLAY = ["bleu", "rouge1", "rouge2", "rougeL", "f1", "macro_f1"]

all_rows = []
for ds_name in all_pre_metrics:
    pre  = all_pre_metrics.get(ds_name, {})
    post = all_post_metrics.get(ds_name, {})
    for m in METRICS_DISPLAY:
        if m not in pre and m not in post:
            continue
        pre_v  = pre.get(m, 0.0)
        post_v = post.get(m, 0.0)
        delta  = post_v - pre_v
        pct    = (delta / pre_v * 100) if pre_v > 0 else float("nan")
        all_rows.append({
            "dataset": ds_name, "metric": m,
            "pre": pre_v, "post": post_v,
            "delta": round(delta, 4),
            "delta_pct": round(pct, 1) if not pd.isna(pct) else None
        })

df_results = pd.DataFrame(all_rows)
df_results.to_csv(RESULTS_DIR / "comparison_full.csv", index=False)

# Per-dataset breakdown
W = 72
for ds_name in all_pre_metrics:
    subset = df_results[df_results["dataset"] == ds_name]
    print(f"\n{'='*W}")
    print(f" {ds_name}")
    print(f"{'-'*W}")
    print(f"  {'Metric':<12} {'Pre-FT':>10} {'Post-FT':>10} {'Δ':>10} {'Δ%':>8}")
    print(f"  {'-'*12} {'-'*10} {'-'*10} {'-'*10} {'-'*8}")
    for _, row in subset.iterrows():
        pct_s = f"{row.delta_pct:+.1f}%" if row.delta_pct is not None else "N/A"
        print(f"  {row.metric:<12} {row.pre:>10.4f} {row.post:>10.4f} "
              f"{row.delta:>+10.4f} {pct_s:>8}")

print(f"\n{'='*W}")
print(f"Outputs saved to {RESULTS_DIR}")
print(f"  comparison_full.csv  — per-dataset per-metric comparison")
print(f"  pre_metrics.json     — raw pre-eval scores (Benchmark A)")
print(f"  post_metrics.json    — raw post-eval scores (Benchmark B)")
print(f"  {LOG_DIR.name}/training_log.csv — step-level loss")
print(f"  {LOG_DIR.name}/eval_log.csv     — per-dataset eval log")

## Cell 14 — Quick Inference Test (Optional)

Reloads the saved LoRA adapter and runs a sanity-check query using the same inference hyperparameters as evaluation.

In [ ]:
# Reload saved adapter for a sanity-check inference
inf_model, inf_tok = load_model_and_tokenizer(lora=False)
inf_model = PeftModel.from_pretrained(inf_model, ADAPTER_DIR)
inf_model.eval()


def ask(question: str, context: str = "General medical knowledge.") -> str:
    """Quick inference using report Appendix A hyperparameters."""
    prompt = (
        f"<instruction>Answer the following medical question clearly and accurately.</instruction>\n"
        f"<context>{context}</context>\n"
        f"<input>{question}</input>\n"
        f"<output>"
    )
    enc = inf_tok(prompt, return_tensors="pt", truncation=True,
                  max_length=MAX_SEQ_LEN - MAX_NEW_TOKENS).to(inf_model.device)
    with torch.no_grad():
        out = inf_model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            repetition_penalty=REP_PENALTY,
            pad_token_id=inf_tok.eos_token_id,
        )
    new = out[0][enc["input_ids"].shape[1]:]
    return extract_generated(inf_tok.decode(new, skip_special_tokens=True))


q = "What is the first-line treatment for type 2 diabetes?"
print(f"Q: {q}")
print(f"A: {ask(q)}")